# Running LDSC and bivariate LDSC with PLDSC

This notebook loads GWAS summary statistics with **allelign**, estimates SNP heritability with LDSC, and estimates genetic correlation with bivariate LDSC. Run its cells in order with the Python environment in which PLDSC and allelign are installed.

The first example creates synthetic data. Its z scores are independent across variants, so it illustrates the workflow; it is **not** a calibration study or a validated reference for real GWAS. For real data, replace the paths and `M` as described below.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import subprocess
import sys

import allelign
import numpy as np
from pldsc import ldsc_h2_sumstats, ldsc_rg_sumstats


## 1. Create example files

Run this notebook from the PLDSC checkout. The example script writes two compressed GWAS files, a sorted PLINK `.bim` reference table, and an LD-score `.npy` vector in the same reference order. The files live in a temporary directory.


In [ ]:
temp = TemporaryDirectory(prefix="pldsc-notebook-")
demo = Path(temp.name) / "demo"
script = Path("examples/ordinary_ldsc.py")
if not script.is_file():
    script = Path("ordinary_ldsc.py")  # notebook opened from examples/
if not script.is_file():
    raise FileNotFoundError("Open this notebook from the PLDSC checkout")
subprocess.run([sys.executable, str(script), "--out-dir", str(demo)], check=True)

trait1_path = demo / "trait1.tsv.gz"
trait2_path = demo / "trait2.tsv.gz"
reference_path = demo / "reference.bim"
scores_path = demo / "reference.ldscores.npy"
M = 4000  # the number of variants summed into each synthetic LD score


## 2. Load and inspect inputs

Allelign recognizes common GWAS column names, including per-variant N, and can read tabular gzip files. Check the recorded N source: it must be appropriate for each study. An explicit numeric `n_eff` replaces all per-variant sample sizes; a string names a column without flattening it.

The reference and the score vector must have the same row order. `M` counts all variants summed into each score, before any GWAS row is filtered. It is not necessarily the number of reference rows or retained GWAS rows. The GWAS and reference must use the same genome build; the code does not lift coordinates over.


In [ ]:
reference = allelign.read_variants(reference_path)
scores = np.load(scores_path, allow_pickle=False)
ss1 = allelign.read_sumstats(trait1_path)
ss2 = allelign.read_sumstats(trait2_path)
print("reference rows / scores:", len(reference), len(scores))
print("trait 1 parse:", ss1.parse_log)
print("trait 2 parse:", ss2.parse_log)


For a real file with nonstandard columns, for example:

```python
ss1 = allelign.read_sumstats("trait1.tsv.gz", ea="EFFECT_ALLELE",
                            oa="OTHER_ALLELE", beta="BETA", se="SE",
                            n_eff="EFFECTIVE_N")
```

If the file has no N column and a constant effective N is justified, use `n_eff=100_000` instead. For a case-control study, check how effective N was defined. In a compatible meta-analysis with differing case fractions, pooled case/control counts can overstate effective N; use an appropriate per-variant effective N or sum of study-specific effective Ns.


## 3. Fit LDSC and bivariate LDSC

The wrappers use allelign to filter invalid effects, low N, duplicates, and available low-MAF/INFO rows; harmonize to the reference alleles; and exclude ambiguous A/T and C/G sites by default. They then apply the LDSC chi-square cap and fit on genomic-order rows. Bivariate LDSC uses the variants retained in **both** traits. Both analyses use the same LD-score vector, which must be suitable for both traits.


In [ ]:
h2 = ldsc_h2_sumstats(ss1, reference, scores, m_snps=M, n_blocks=40)
rg = ldsc_rg_sumstats(ss1, ss2, reference, scores, m_snps=M,
                      n_blocks=40, ncores=2)
print("Trait 1 h²:", h2.estimate.h2, "SE:", h2.estimate.h2_se,
      "approx. 95% CI:", h2.estimate.h2_ci)
print("Genetic correlation:", rg.estimate.rg, "SE:", rg.estimate.rg_se)
print("Genetic covariance:", rg.estimate.gcov,
      "cross-trait intercept:", rg.estimate.gcov_intercept)


In [ ]:
print("Trait 1 file QC:", rg.qc[0])
print("Trait 2 file QC:", rg.qc[1])
print("Trait 1 alignment:", rg.alignment[0])
print("Trait 2 alignment:", rg.alignment[1])
print("Common before chi-square cap:", rg.n_common,
      "removed by cap:", rg.n_chi2_dropped,
      "retained:", len(rg.reference_index))


Inspect the logs for unexpected loss, sample-size source, allele mismatches, and sign flips before interpreting the estimates. Use `qc_options` for h², or `qc_options1` and `qc_options2` for bivariate LDSC, to change allelign thresholds deliberately.

For example, `keep, log = allelign.qc_sumstats(ss1)` previews the default file QC. Passing the original `ss1` to the wrappers lets them perform that QC once for the analysis and retain its full log. Passing an already QC-filtered table causes QC to run again, and the relative-N threshold may be recalculated on the filtered table.


## 4. Optional: exclude long-range LD regions

Long-range LD exclusions are not automatic. If the GWAS coordinates are GRCh37 and this selection is appropriate for the analysis, allelign supplies a built-in list. Decide separately whether to include the APOE locus. Supply a build-appropriate custom region list for other builds; never apply the GRCh37 list to GRCh38 coordinates.


In [ ]:
inside1 = allelign.in_long_range_ld(ss1.chrom, ss1.pos,
                                    genome_build="GRCh37", include_apoe=True)
inside2 = allelign.in_long_range_ld(ss2.chrom, ss2.pos,
                                    genome_build="GRCh37", include_apoe=True)
print("Rows in selected regions:", int(inside1.sum()), int(inside2.sum()))
rg_without_regions = ldsc_rg_sumstats(ss1.subset(~inside1), ss2.subset(~inside2),
                                      reference, scores, m_snps=M,
                                      n_blocks=40, ncores=2)
print("r_g after exclusion:", rg_without_regions.estimate.rg)


## 5. Interpretation and real data

These estimates are not clipped: negative h² or r_g outside [-1, 1] can occur. An undefined r_g or standard error is returned as NaN. The cross-trait intercept may reflect sample overlap, correlated confounding, or model and reference mismatch; it cannot identify their separate contributions. Binary-trait h² is not automatically transformed to the liability scale.

For your data, set the four input paths and `M`, then rerun from **2. Load and inspect inputs**. Confirm ancestry/reference compatibility, genome build, score-row identities, score construction and sample-size definitions first. See [`docs/LDSC.md`](../docs/LDSC.md) for command-line use, input formats and assumptions, and [`report/methods.pdf`](../report/methods.pdf) for the methods.
